Merge Daily Bronze Update
Reads the latest daily file(s) produced by Data Factory, appends them to the full history, dedupes, saves.

**Input**: bronze/live/daily/*.json (produced daily by Data Factory)
**Output**: bronze/live/battery_full_history.json (updated)

In [0]:
%run ./_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_bronze
import pandas as pd

blob_service = get_blob_service(storage_account_name, storage_account_key)

Read existing full history

In [0]:
full_history = read_bronze(blob_service, "live/battery_full_history.json")
print(f"Existing history: {len(full_history)} rows, max date: {pd.to_datetime(full_history['postingDate']).max()}")

Read any new daily files

In [0]:
container_client = blob_service.get_container_client("bronze")
daily_blobs = sorted(container_client.list_blobs(name_starts_with="live/daily/"), key=lambda b: b.name)

print(f"Found {len(daily_blobs)} daily files")

new_batches = []
for blob in daily_blobs:
    day_df = read_bronze(blob_service, blob.name)
    new_batches.append(day_df)
    print(f"  {blob.name}: {len(day_df)} rows")

In [0]:
new_batches = []
for blob in daily_blobs:
    day_df = read_bronze(blob_service, blob.name)
    new_batches.append(day_df)
    print(f"  {blob.name}: {len(day_df)} rows")

In [0]:
full_history = read_bronze(blob_service, "live/battery_full_history.json")
print(f"Existing history: {len(full_history)} rows, max date: {pd.to_datetime(full_history['postingDate']).max()}")

In [0]:
if new_batches:
    combined = pd.concat([full_history] + new_batches, ignore_index=True)
    combined = combined.drop_duplicates()
    print(f"Combined total: {len(combined)} rows (was {len(full_history)})")
else:
    combined = full_history
    print("No new daily files found — history unchanged")

payload = combined.to_json(orient="records", date_format="iso", lines=False)
blob_client = blob_service.get_blob_client(container="bronze", blob="live/battery_full_history.json")
blob_client.upload_blob(payload, overwrite=True)
print(f"Saved updated history: {len(combined)} rows, max date: {pd.to_datetime(combined['postingDate']).max()}")

In [0]:
for blob in daily_blobs:
    blob_client = blob_service.get_blob_client(container="bronze", blob=blob.name)
    blob_client.delete_blob()
print(f"Deleted {len(daily_blobs)} processed daily files")

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob="live/forecasts/overall_forecast_latest.json")
stream = blob_client.download_blob().readall()
import json
print(json.loads(stream))

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob="live/forecasts/overall_forecast_latest.json")
stream = blob_client.download_blob().readall()
import json
print(json.loads(stream))